# Cleaning of geoapify json response

In [37]:
import json
from pathlib import Path
import pandas as pd

In [ ]:
RAW_FILE = "../raw/venues_raw.json"
CLEAN_JSON = "../cleaned/venues_cleaned.json"
CLEAN_CSV = "../cleaned/venues_cleaned.csv"

with open(RAW_FILE, "r", encoding="utf-8") as file:
    raw_data = json.load(file)

print(f"Raw records loaded: {len(raw_data)}")

df = pd.json_normalize(raw_data)


Raw records loaded: 400
(400, 392)
Index(['type', 'properties.name', 'properties.country',
       'properties.country_code', 'properties.state', 'properties.city',
       'properties.postcode', 'properties.suburb', 'properties.quarter',
       'properties.street',
       ...
       'properties.datasource.raw.min_height',
       'properties.datasource.raw.payment:cards',
       'properties.datasource.raw.roof:material',
       'properties.payment_options.cards', 'properties.datasource.raw.leisure',
       'properties.datasource.raw.max_age', 'properties.restrictions.max_age',
       'properties.datasource.raw.barrier',
       'properties.datasource.raw.fence_type',
       'properties.datasource.raw.surface'],
      dtype='str', length=392)


In [57]:
print(df.shape)
print(df.columns)
print(df.info())

(318, 10)
Index(['geoapify_place_id', 'name', 'address', 'postcode', 'borough',
       'latitude', 'longitude', 'website', 'opening_hours', 'category'],
      dtype='str')
<class 'pandas.DataFrame'>
Index: 318 entries, 0 to 394
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   geoapify_place_id  318 non-null    str    
 1   name               318 non-null    str    
 2   address            318 non-null    str    
 3   postcode           318 non-null    string 
 4   borough            197 non-null    str    
 5   latitude           318 non-null    float64
 6   longitude          318 non-null    float64
 7   website            179 non-null    str    
 8   opening_hours      112 non-null    str    
 9   category           318 non-null    str    
dtypes: float64(2), str(7), string(1)
memory usage: 27.3 KB
None


### The below code is to save the starting count

In [39]:
len(df)

400

### Save the fields that we actually need

In [40]:
columns2 = [
    "properties.place_id",
    "properties.name",
    "properties.categories",
    "properties.formatted",
    "properties.postcode",
    "properties.district",
    "properties.lat",
    "properties.lon",
    "properties.website",
    "properties.opening_hours"
]

df = df.reindex(columns=columns2) # Makes the dataframes columns match the order and names in the columns2 list

### Rename the columns to match the database fields:

In [41]:
df = df.rename(columns={
    "properties.place_id": "geoapify_place_id",
    "properties.name": "name",
    "properties.categories": "categories",
    "properties.formatted": "address",
    "properties.postcode": "postcode",
    "properties.district": "borough",
    "properties.lat": "latitude",
    "properties.lon": "longitude",
    "properties.website": "website",
    "properties.opening_hours": "opening_hours"
})

### Clean the category field to match the database

- Database schema only has one category field wheras API returns multiple.

In [42]:
def get_category(categories):
    if isinstance(categories, list) and len(categories) > 0:
        return categories[-1]

    return None

df["category"] = df["categories"].apply(get_category)

df = df.drop(columns=["categories"])
df.head()

,geoapify_place_id,name,address,postcode,borough,latitude,longitude,website,opening_hours,category
0,5184503981446ec0bf59d3009475eec04940f00103f901...,Caffè Nero,"Caffè Nero, 60-61 Trafalgar Square, London, WC...",WC2N 5DS,NaN,51.507277,-0.128365,https://www.caffenero.com/uk/stores/trafalgar-sq,Mo-Fr 06:30-21:00; Sa 07:00-21:00; Su 07:30-20:00,vegetarian
1,519008c109d937c0bf593039a0abe8c04940f00102f901...,Caffe Concerto,"Caffe Concerto, 4-5 Northumberland Avenue, Lon...",WC2N 5BW,NaN,51.507116,-0.126693,https://www.caffeconcerto.co.uk/,07:00-23:00,wheelchair.limited
2,510b0c59ddea39c0bf59111ac1c6f5c04940f00103f901...,Costa,"Costa, 1 Strand, London, WC2N 5NJ, United Kingdom",WC2N 5NJ,NaN,51.507500,-0.126768,NaN,NaN,wheelchair.yes
3,5160feafdfb831c0bf59c471851c00c14940f00103f901...,The Grand Caffè,"The Grand Caffè, 1 Strand, London, WC2N 5NJ, U...",WC2N 5NJ,NaN,51.507816,-0.126517,NaN,NaN,catering.cafe
4,51bd9b5dadc909c0bf598be4863a07c14940f00103f901...,Costa,"Costa, Hungerford Lane, London, WC2N 5NG, Unit...",WC2N 5NG,NaN,51.508033,-0.125299,NaN,NaN,wheelchair.yes


### Removal of null values
- The below code shows the venues that have null values and the cell block after that removes these values.


In [47]:
df[df['latitude'].isnull() | df['longitude'].isnull() | df['name'].isnull() | df['geoapify_place_id'].isnull()]

,geoapify_place_id,name,address,postcode,borough,latitude,longitude,website,opening_hours,category
157,512d4c658a700cc1bf597f446cd3f8c04940f00102f901...,NaN,"12 Waterloo Place, London, SW1Y 4AR, United Ki...",SW1Y 4AR,St. James's,51.507594,-0.133192,NaN,NaN,catering.restaurant
300,51175b54f4e196bebf59c9dfda4c38c14940f00102f901...,NaN,"Victoria Embankment, London, WC2N 6PB, United ...",WC2N 6PB,St Clement Danes,51.509489,-0.119522,NaN,NaN,wheelchair.limited
302,5168cf09f87f74c0bf5915645a1ee7c14940f00102f901...,NaN,"St. Giles Passage, London, WC2H 8DE, United Ki...",WC2H 8DE,London Borough of Camden,51.514884,-0.128565,NaN,NaN,leisure.playground
304,5115ff412497cabebf59dd9d0053d0c14940f00102f901...,NaN,"Drury Lane, London, WC2B 5SQ, United Kingdom",WC2B 5SQ,St Clement Danes,51.514170,-0.120279,NaN,NaN,leisure.playground
305,515806646c79b6c1bf59e6cda02124c04940f00102f901...,NaN,"Birdcage Walk, London, SW1H 9AP, United Kingdom",SW1H 9AP,NaN,51.501100,-0.138368,NaN,NaN,leisure.playground
...,...,...,...,...,...,...,...,...,...,...
395,514411d9ac9fa3b9bf59c217265305bf4940f00102f901...,NaN,"Hampton Street, London, SE1 6SL, United Kingdom",SE1 6SL,London Borough of Southwark,51.492350,-0.100153,NaN,NaN,leisure.playground
396,51b28a2a01f96cbbbf5923e1f0d153c34940f00102f901...,NaN,"Northampton Road, London, EC1R 0HU, United Kin...",EC1R 0HU,London Borough of Islington,51.526011,-0.107122,NaN,NaN,leisure.playground
397,51d73638d5f2bcb8bf59a4fc65017bc24940f00102f901...,NaN,"Thomas More Highwalk, City of London, EC2Y 8BT...",EC2Y 8BT,NaN,51.519379,-0.096633,NaN,NaN,leisure.playground
398,5197497b24d37ac2bf59a1f5251852be4940f00102f901...,NaN,"Lupus Street, London, SW1V 3HD, United Kingdom",SW1V 3HD,NaN,51.486935,-0.144378,NaN,NaN,leisure.playground


In [48]:
df = df.dropna(subset=[
    'latitude',
    'longitude',
    'name',
    'geoapify_place_id'
])

- Now the same code that checks for null values returns no results.

In [49]:
df[df['latitude'].isnull() | df['longitude'].isnull() | df['name'].isnull() | df['geoapify_place_id'].isnull()]

,geoapify_place_id,name,address,postcode,borough,latitude,longitude,website,opening_hours,category


- No duplicated valeus:

In [52]:
df[df.duplicated(keep=False)]

,geoapify_place_id,name,address,postcode,borough,latitude,longitude,website,opening_hours,category


### Clean venue names
- Remove white spaces in front or trailing

In [53]:
df["name"] = df["name"].str.strip()

### Clean postcodes

In [55]:
df["postcode"] = (
    df["postcode"].str.strip().str.upper()
)

In [60]:
print(df.info())
print(df.shape)
print(len(df))

<class 'pandas.DataFrame'>
Index: 318 entries, 0 to 394
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   geoapify_place_id  318 non-null    str    
 1   name               318 non-null    str    
 2   address            318 non-null    str    
 3   postcode           318 non-null    string 
 4   borough            197 non-null    str    
 5   latitude           318 non-null    float64
 6   longitude          318 non-null    float64
 7   website            179 non-null    str    
 8   opening_hours      112 non-null    str    
 9   category           318 non-null    str    
dtypes: float64(2), str(7), string(1)
memory usage: 27.3 KB
None
(318, 10)
318


### Export cleaned data to the cleaned data folder:

In [61]:
df.to_json(
    CLEAN_JSON,
    orient="records",
    indent=2,
    force_ascii=False
)

df.to_csv(
    CLEAN_CSV,
    index=False
)